Step 1 — Load Raw Data
Let's bring in the raw WHO dataset and check its dimensions to make sure everything loaded properly.

In [3]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("../data/health_indicators_nga.csv", encoding="utf-8-sig")

print(f"Raw dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Raw dataset: 33263 rows x 17 columns


,GHO (CODE),GHO (DISPLAY),GHO (URL),YEAR (DISPLAY),STARTYEAR,ENDYEAR,REGION (CODE),REGION (DISPLAY),COUNTRY (CODE),COUNTRY (DISPLAY),DIMENSION (TYPE),DIMENSION (CODE),DIMENSION (NAME),Numeric,Value,Low,High
0,LIFE_0000000035,ex - expectation of life at age x,https://www.who.int/data/gho/data/indicators/i...,2016,2016,2016,AFR,Africa,NGA,Nigeria,SEX,SEX_MLE,Male,42.058859,42.05886,NaN,NaN
1,LIFE_0000000030,nqx - probability of dying between ages x and x+n,https://www.who.int/data/gho/data/indicators/i...,2013,2013,2013,AFR,Africa,NGA,Nigeria,SEX,SEX_BTSX,Both sexes,0.045697,0.04570,NaN,NaN
2,TOBACCO_MPOWER_E3_INDIRECTADBANSPLUS,"Tobacco Enforce bans: E3 indirect advertising,...",https://www.who.int/data/gho/data/indicators/i...,2020,2020,2020,AFR,Africa,NGA,Nigeria,TOBACCO_INDICATOR,TOBACCO_INDICATOR_E17b_csr_promo_self,Ban on tobacco companies/the tobacco industry ...,NaN,Yes,NaN,NaN
3,SA_0000001400,"Alcohol, recorded per capita (15+) consumption...",https://www.who.int/data/gho/data/indicators/i...,1969,1969,1969,AFR,Africa,NGA,Nigeria,ALCOHOLTYPE,ALCOHOLTYPE_SA_WINE,Wine,0.000000,0,NaN,NaN
4,CM_01,Number of under-five deaths,https://www.who.int/data/gho/data/indicators/i...,2002,2002,2002,AFR,Africa,NGA,Nigeria,SEX,SEX_BTSX,Both sexes,196199.000000,196 199 [182 290-211 064],182290.0,211064.0


Step 2 — Filter Child Health Data
et's filter it down to focus strictly on child nutrition and immunization metrics, stripping out unrelated records like adult health, maternal indicators, and tobacco usage.

In [4]:
keywords = [
    'immuniz', 'vaccin', 'bcg', 'hepb', 'dtp', 'measles', 'mcv', 
    'polio', 'ipv', 'rotavirus', 'stunt', 'wast', 'underweight', 
    'mortality', 'infant', 'under-five', 'child', 'neonatal', 'breastfeeding'
]
pattern = '|'.join(keywords)

# Filter child indicators and remove adult/tobacco data
df_child = df[df['GHO (DISPLAY)'].str.contains(pattern, case=False, na=False)].copy()
df_child = df_child[~df_child['GHO (DISPLAY)'].str.contains('adult|maternal|tobacco', case=False, na=False)]

print(f"Filtered dataset: {df_child.shape[0]} rows x {df_child.shape[1]} columns")
df_child.head()

Filtered dataset: 10271 rows x 17 columns


,GHO (CODE),GHO (DISPLAY),GHO (URL),YEAR (DISPLAY),STARTYEAR,ENDYEAR,REGION (CODE),REGION (DISPLAY),COUNTRY (CODE),COUNTRY (DISPLAY),DIMENSION (TYPE),DIMENSION (CODE),DIMENSION (NAME),Numeric,Value,Low,High
4,CM_01,Number of under-five deaths,https://www.who.int/data/gho/data/indicators/i...,2002,2002,2002,AFR,Africa,NGA,Nigeria,SEX,SEX_BTSX,Both sexes,196199.000000,196 199 [182 290-211 064],182290.000000,211064.000000
7,MORT_300,Distribution of causes of death among children...,https://www.who.int/data/gho/data/indicators/i...,2009,2009,2009,AFR,Africa,NGA,Nigeria,AGEGROUP,AGEGROUP_MONTHS1-59,1-59 months,0.000130,0,NaN,NaN
9,NCD_BMI_MINUS2C,"Thinness among children and adolescents, BMI-f...",https://www.who.int/data/gho/data/indicators/i...,2017,2017,2017,AFR,Africa,NGA,Nigeria,SEX,SEX_BTSX,Both sexes,14.553297,14.6 [11.8-17.5],11.837688,17.493656
16,NUTRITION_ANAEMIA_CHILDREN_NUM,Number of children aged 6-59 months with anaem...,https://www.who.int/data/gho/data/indicators/i...,2011,2011,2011,AFR,Africa,NGA,Nigeria,SEX,SEX_BTSX,Both sexes,20516.733000,20517 [18820-21857],18820.000000,21857.000000
22,MORT_100,"Number of deaths in children aged <5 years, by...",https://www.who.int/data/gho/data/indicators/i...,2006,2006,2006,AFR,Africa,NGA,Nigeria,SEX,SEX_BTSX,Both sexes,42520.704050,42520.7,NaN,NaN


Step 3 — Select and Rename Columns
Let's keep only the columns we need for modeling and rename them to simpler, cleaner names.

In [5]:
cols_to_keep = [
    'GHO (CODE)', 'GHO (DISPLAY)', 'YEAR (DISPLAY)', 
    'DIMENSION (NAME)', 'Numeric', 'Low', 'High'
]

df_clean = df_child[cols_to_keep].copy()

df_clean.rename(columns={
    'GHO (CODE)': 'indicator_code',
    'GHO (DISPLAY)': 'indicator_name',
    'YEAR (DISPLAY)': 'year',
    'DIMENSION (NAME)': 'gender_dimension',
    'Numeric': 'value',
    'Low': 'value_low',
    'High': 'value_high'
}, inplace=True)

print(f"Cleaned dataset: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
df_clean.head()

Cleaned dataset: 10271 rows x 7 columns


,indicator_code,indicator_name,year,gender_dimension,value,value_low,value_high
4,CM_01,Number of under-five deaths,2002,Both sexes,196199.000000,182290.000000,211064.000000
7,MORT_300,Distribution of causes of death among children...,2009,1-59 months,0.000130,NaN,NaN
9,NCD_BMI_MINUS2C,"Thinness among children and adolescents, BMI-f...",2017,Both sexes,14.553297,11.837688,17.493656
16,NUTRITION_ANAEMIA_CHILDREN_NUM,Number of children aged 6-59 months with anaem...,2011,Both sexes,20516.733000,18820.000000,21857.000000
22,MORT_100,"Number of deaths in children aged <5 years, by...",2006,Both sexes,42520.704050,NaN,NaN


Let's save our cleaned child health dataset into the data/ folder for the next phase of the project.

In [6]:
# Save cleaned file
df_clean.to_csv("../data/who_cleaned_child_health.csv", index=False, encoding="utf-8-sig")

print("Successfully saved cleaned dataset!")

Successfully saved cleaned dataset!
